# Priorización de cobranza con Lending Club
Este notebook utiliza las funciones documentadas de `priorizacion_recobro.py`. Ejecuta las celdas en orden desde la carpeta del proyecto.

**Alcance:** `prob_recobro` aproxima el desenlace Fully Paid entre cuentas cerradas. No acredita regularización tras mora ni respuesta a una llamada. Los estados corresponden al corte histórico del archivo, no a hoy. La prioridad monetaria es experimental.

## 1. Dependencias y configuración
Instala `requirements.txt` en el entorno del kernel si es necesario. La lectura usa seis columnas; `id_col="id"` agrega el identificador para cruzar con el CRM. El CSV comprimido no necesita descomprimirse.

In [ ]:
# Si faltan dependencias, descomenta y ejecuta:
# %pip install -r requirements.txt
from pathlib import Path
import logging
import pandas as pd
from IPython.display import display
from priorizacion_recobro import Config, ejecutar, exportar, localizar_csv

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
config = Config(
    csv_path=None,  # o Path(r'C:\ruta\accepted_2007_to_2018Q4.csv.gz')
    chunksize=100_000,
    max_historicos=200_000,
    arboles=120,
    jobs=2,
    seed=42,
    id_col='id',  # None para seleccionar solo las seis columnas
)
print(localizar_csv(config.csv_path))

## 2. Carga, segmentación y entrenamiento
Se recorre todo el archivo por bloques. Se guarda una muestra uniforme de cerrados y la totalidad de los estados Late (31-120 days) e In Grace Period. Fully Paid=1; Charged Off=0.

Los predictores son delinq_2yrs, revol_util y annual_inc. Último pago y saldo de cierre se excluyen para evitar fuga. La mediana se aprende dentro del pipeline; se reserva un test del 20 % y se calibra en el resto con tres particiones. La ejecución puede tardar varios minutos.

In [ ]:
resultado = ejecutar(config)
display(pd.Series(resultado.metricas['datos'], name='resumen'))

## 3. Evaluación en cuentas cerradas
AUC y average precision: mayor es mejor. Brier y log loss: menor es mejor. Compara con el baseline de prevalencia. Esta evaluación no demuestra desempeño en morosos ni en períodos futuros.

In [ ]:
evaluacion = resultado.metricas['evaluacion']
display(pd.DataFrame({
    'modelo': evaluacion['modelo'],
    'baseline': evaluacion['baseline_prevalencia'],
}))
display(resultado.calibracion)
display(resultado.calidad)

## 4. Scoring e impacto financiero
`prioridad_monetaria = prob_recobro * out_prncp`. Se ordena de mayor a menor, manteniendo el saldo original válido. El saldo no se imputa. Cuentas con saldo inválido/no positivo o sin ningún predictor pasan a revisión.

In [ ]:
df_priorizado = resultado.priorizadas
display(df_priorizado.head(20))
print(f'Cuentas priorizadas: {len(df_priorizado):,}')
print(f'Cuentas para revisión: {len(resultado.revision):,}')

## 5. Exportación
Genera CSV en UTF-8-BOM y métricas JSON. Cambia a `excel=True` si quieres también un libro Excel (requiere openpyxl). Sobrescribe las salidas del mismo directorio.

In [ ]:
exportar(resultado, Path('outputs'), excel=False)

## 6. Roll rates: requiere historial mensual adicional
Este CSV no permite calcular roll rates: se necesitan cortes mensuales de las mismas cuentas con días de mora y saldo. No se generan curvas inventadas. Cuando dispongas de un panel real con id_cuenta, fecha_corte, dias_mora y saldo, usa `calcular_roll_rates` y `graficar_roll_rates` de `roll_rates.py`. Consulta README.md para el contrato de datos y el tratamiento de cuentas sin seguimiento.